In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [2]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
    """Given two integers a and b, this tool returns their product"""
    
    return a * b

In [3]:
print(multiply.invoke({'a':3,'b':4}))

12


In [4]:
print(multiply.name)
print(multiply.args)

multiply
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### tool binding

binding 2 or more tools :

In [5]:
llm = ChatOpenAI()

In [6]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3K12J77Ik5pwcMc3EiTeXOPEWn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7aa9b818-72a6-47ca-b04e-44b9ed0ab490-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
llm_with_tools = llm.bind_tools([multiply]) #  if 2 tools are there then add them to the list by , seperated

### NOTE : Every LLM doesnot have the functionality of tool binding

In [8]:
llm_with_tools.invoke('hi how are you')


AIMessage(content="Hello! I'm here and ready to assist you. How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 58, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3LapPCpts1MWv47sm7ooK9NG11', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--319c3d75-9023-46f8-845a-5d482388c893-0', usage_metadata={'input_tokens': 58, 'output_tokens': 19, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
query = HumanMessage('can you multiply 3 with 10')

In [10]:
message = [query]

In [11]:
message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

## tool call

In [12]:
# tool call
result = llm_with_tools.invoke(message)
result


AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3MhQ2pSlGy7PKOSagIat0g80cG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--30e6d55e-3b1c-440b-abb4-27bf89ad10a6-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'type': 'tool_call'}], usage_metadata={'input_tokens': 62, 'output_tokens': 17, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 

In [13]:
# result.tool_calls[0]

In [14]:
message.append(result)

In [15]:
message
# it has both human and AI message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3MhQ2pSlGy7PKOSagIat0g80cG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--30e6d55e-3b1c-440b-abb4-27bf89ad10a6-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'type': 'tool_call'}], usage_metadata={'input_tokens': 6


### NOTE : LLM can never call the tool, it can only suggest the tool

In [16]:
# passing the iput of llm to the tool
multiply.invoke(
    {'name': 'multiply',
     'args': {'a': 3, 'b': 10},
     'id': 'call_U9VWkwh2gmuqtQCJji0J6rQO',
     'type': 'tool_call'}
)

ToolMessage(content='30', name='multiply', tool_call_id='call_U9VWkwh2gmuqtQCJji0J6rQO')

#### Tool Message is a special message given by tool call which can be sent to the LLM

In [17]:
# tool execution
tool_result = multiply.invoke(result.tool_calls[0]) # instead of the arguments we are sending entire tool call
tool_result

ToolMessage(content='30', name='multiply', tool_call_id='call_PmC8E6VzrcxUnvcxZcimELzE')

In [18]:
message.append(tool_result) # appending tool_result in the message list

In [19]:
message

# has 3 things : 
# 1. Human message
# 2. Ai messgae
# 3. Tool message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3MhQ2pSlGy7PKOSagIat0g80cG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--30e6d55e-3b1c-440b-abb4-27bf89ad10a6-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_PmC8E6VzrcxUnvcxZcimELzE', 'type': 'tool_call'}], usage_metadata={'input_tokens': 6

In [20]:
llm_with_tools.invoke(message)

AIMessage(content='The product of 3 and 10 is 30.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 87, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C9G3N38xKjLoGLp2ni5Raq6NvL6Hb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--6c1d6065-0f0b-4127-a910-45c7429eab68-0', usage_metadata={'input_tokens': 87, 'output_tokens': 13, 'total_tokens': 100, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})